<a href="https://colab.research.google.com/github/sharc-jx/ao3stats/blob/main/ao3Stats.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Analyse My Ao3 History

**Code written by:** nyxbda (updated 27/12/2023)

I wanted a programme to analyse my Ao3 history, and be able to take a comprehensive and in-depth view of my statistics because I am a data nerd.

To run this code, just hit run all (`Runtime -> Run all` or `F9`)- or you can run cell by cell by clicking on the play button on the side of each cell/cell group (you can expand the code). Please note at certain points you will need to input things (ie. username/password or when you want to run the analysis) so you'll need to check that you've done that before the analysis will continue.

I have private outputs enabled on this notebook, and it is read only, meaning that your variables (any info you enter into this notebook), outputs or any files created will not be saved (will not be viewable to anyone else, including me). Everytime you restart the code (restart the kernel, including when you exit the programme) all data is lost, and you will need to download any files or graphs if you want to save them for later. If you want to be extra sure your data does not leave your hands, you can save this notebook to your own personal Google Drive or your desktop (make sure you have a programme that can run Python like [Anaconda](anaconda.org) or [Visual Studio Code](https://code.visualstudio.com/)) to run the code there.

I have commented this code so if you are not used to coding or Python you can (hopefully) follow the steps, but do Google and look up some of the commands I use if you are not sure about how it works! I would love for this to be a gateway for more people learning (at least) the basics of coding, and this means you have even more control over your own data.

**A summary of this code:**
- cell 1-2: imports the necessary code libraries/modules to run (if you haven't run all you can just hit play next to the 2 hidden cells)
- Part 1: run all of this section to scrape your data from Ao3 and clean your data to make a neat table- use this if you don't have a presaved .csv file of your Ao3 history - you can run all the cells one by one of either part (a) or part (b) - make sure you fill in the input fields when prompted
    - (a) this will scrape your data and transform it all in the code, so you don't need to do anything else!
    - (b) if you'd rather, you can use a separate webscraper in your browser, downlaod that .csv file and then tidy the data in this part (more info on this in part (b))
- Part 2: if you haven't run all, you must run all of this section (the 9 hidden cells which contain all of the functions that make the graphs) so that the analysis can run
- Part 3: the analysis!

Also if there are certain things you would be interested in being analysed that I haven't covered, or if you've found bugs or know other ways to optimise the code, please let me know! While I'll try to stay on top of bug fixes/code optimisations, I may not check in that regularly and I cannot guaruntee I will add any changes/additional stats people are interested in. I'll try my best though, and I hope that you'll find these are already quite comprehensive!

## Get Necessary Packages

In [ ]:
### GET PACKAGE DEPENDENCIES
!pip install bs4 getpass4 #web-scraping packages
!pip install requests[security]
!pip install pandas numpy #useful packages
!pip install matplotlib IPython ipywidgets ipympl #graphing packages

In [ ]:
### IMPORT THE LIBRARIES NEEDED FOR THE CODE TO RUN

#for web-scraping
from bs4 import BeautifulSoup as bs
import requests
import time
import re
import getpass

#for general analysis/generally useful modules
import pandas as pd
pd.options.mode.chained_assignment = None
from pandas import DataFrame as df
import numpy as np
from ast import literal_eval

#for graphing
import matplotlib.pyplot as plt
from matplotlib.ticker import (MultipleLocator, AutoMinorLocator)
from IPython.display import display
from ipywidgets import interact, fixed
import ipywidgets as widgets
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"
plt.rcParams['axes.axisbelow'] = True
pd.set_option('display.max_colwidth', None)
%matplotlib inline

## Step One - Getting Your Data (Scraping) from Ao3

### (a) Scraping Your Data Within the Code
Scraping code originally from [klipklapper](https://www.reddit.com/r/AO3/comments/zs5w96/ao3_wrapped/), [originally found here](https://colab.research.google.com/drive/179YXx5EwkB3y696R06m1Kp-eMwg1l9YR?usp=sharing#scrollTo=2jySAPu6392O), an extension of [teresachenec's wrapped from 2021](https://github.com/teresachenec/ao3wrapped).

Now further optimised for 2023 Ao3 API and made to run faster.

If you do not want to use this code to get your data from Ao3, skip to part (b) where I can tell you how to download it yourself from your browser.

**How this code works works:**
- you must be logged in so that the code can access your personal history for the stats - which is the whole point! - so you must enter your username and password
    - however, as stated before, all information is saved locally when you run the code, meaning neither the output nor files created are saved nor visible to anyone else
    - for further security, your username and password is saved in a function, so that once the function is done running (ie. once I have pulled the Ao3 history pages), your password is no longer stored (you can check to make sure it is read-only)
    - additionally, the `getpass` library is used to ensure your user input is hidden instead of shown like normal inputs (as seen when entering username)
- the code employs a web scraper from the Python package `Beautiful Soup` which will access the HTML (back-end code) of the history webpages of your Ao3 to pull the information about the fics you have read such as title etc
    - to learn more about web scrapers work see [here](https://www.geeksforgeeks.org/what-is-web-scraping-and-how-to-use-it/)
    - to learn more about Beautiful Soup check out the documentation [here](https://beautiful-soup-4.readthedocs.io/en/latest/)


In [ ]:
def ao3login():
    """A function to connect to your Ao3 account. Only the page is returned once you have been logged in."""
    with requests.Session() as s:
        # Get user information
        username = input('Please input your Ao3 username: ')
        password = getpass.getpass(prompt='Please input your password: ')
        # Get information from the login page to allow connection
        req = s.get('https://archiveofourown.org/users/login').text #get information about the login page
        html = bs(req,"html.parser") #read the page
        token = html.find("input", {"name": "authenticity_token"}).attrs["value"] #to determine the authenticity token (to allow it to connect)
        # Creates payload to login to ao3 (the information to send to the website to allow login)
        payload = {"utf8": "✓",
                    "authenticity_token": token,
                    "user[login]": username,
                    "user[password]": password,
                    "commit": "Log+In"}
        # POST (send) request to ao3 to login
        p = s.post("https://archiveofourown.org/users/login", data=payload)
    return username,s,p

In [ ]:
def getWorkStats(w):
    """Get all the information about a particular work."""

    title = w.find("div", {"class": "header module"}).find("h4", {"class": "heading"}).find("a").text #get title information
    url = 'https://archiveofourown.org' + w.find("div", {"class": "header module"}).find("h4", {"class": "heading"}).find("a", href=True)['href']

    categories = w.find("div", {"class": "header module"}).find("ul", {"class": "required-tags"}).text.split('\n ') #get the header information to determine...
    warning = categories[2].split(', ') #warnings
    rating = categories[1] #rating
    category = categories[3].split(', ') #ship type
    complete = categories[4].replace('\n','') #completion status

    updated = w.find("div", {"class": "header module"}).find("p").text #determine when it was last updated

    viewed = w.find("div", {"class": "user module group"}).find("h4").text.split('\nVisited ') #determine when and how often the fic was viewed
    last_visited = viewed[0].replace('\nLast visited: ', '').replace('\n(Latest version.)', '').replace('\n(Minor edits made since then.)','').replace('\n(Update available.)','') #clean up the visitation information (remove other info to just leave the date)
    visitations = viewed[1].replace(' times\n', '') #get just the visitation information
    if 'Marked for Later' in visitations: #note if a work is in your "Marked for Later" or not
        tbr = True
        visitations = visitations.replace('(Marked for Later.)\n', '')
    else: tbr = False
    if(visitations == "once\n"): visitations = 1 #and turn to a number instead of a string
    else: visitations = int(visitations)

    words = int(w.find("dl", {"class": "stats"}).find("dd", {"class": "words"}).text.replace(",", "")) #get the number of words

    authors = [] #get a list of the authors
    for author in w.find("div", {"class": "header module"}).find("h4", {"class": "heading"}).find_all(rel="author"): authors.append(author.text)

    fandoms = [] #get a list of the fandoms
    for fandom in w.find("div", {"class": "header module"}).find("h5", "fandoms heading").find_all("a"): fandoms.append(fandom.text)

    ships = [] #get a list of the ships
    for ship in w.find("ul", {"class": "tags commas"}).find_all("li", {"class": "relationships"}): ships.append(ship.text)

    characters = [] #get a list of the characters
    for character in w.find("ul", {"class": "tags commas"}).find_all("li", {"class": "characters"}): characters.append(character.text)

    additional_tags = [] #get a list of the tags
    for tag in w.find("ul", {"class": "tags commas"}).find_all("li", {"class": "freeforms"}): additional_tags.append(tag.text)

    summary = ''
    if w.find("blockquote", {"class": "userstuff summary"}): summary = w.find("blockquote", {"class": "userstuff summary"}).text.strip() #pull the summary of the fic

    data = [title, url, tbr, False, warning, rating, category, updated, last_visited, visitations, words, complete, authors, fandoms, ships, characters, additional_tags, summary] #array of all your data

    return data

In [ ]:
nativeScrape = input('Would you like to scrape your Ao3 data in this code? (y/n): ') #if someone has hit 'run all' checks you want to perform each part of the code

In [ ]:
if nativeScrape == 'y' or nativeScrape == 'yes': # if you would like to scrape via the code in this notebook
    ao3History = df(columns=["title", "url", "tbr", "deleted", "warning", "rating", "ship type", "updated", "viewed", "times viewed", "wordcount", "completion status", "authors", "fandoms", "relationships", "characters", "tags", "summary"]) #create a dataframe to hold all the info about the fics you have read
    pages = input('Please enter the number of pages in your Ao3 history: ') #determine the number of pages to cycle through
    username,s,p = ao3login() #login to the first page

In [ ]:
if nativeScrape == 'y' or nativeScrape == 'yes': # if you would like to scrape via the code in this notebook - do the scraping!
    for page in range(1,int(pages)+1): #cycle through the number of pages

        r = s.get(f"https://archiveofourown.org/users/{username}/readings/?page={page}") #load the page
        soup = bs(r.content, "html.parser") #get all the content of the page

        if soup.find("div", {"class": "flash error"}) != None: #if the content isn't there, raise an error message
            if soup.find("div", {"class": "flash error"}).text == "Sorry, you don't have permission to access the page you were trying to reach. Please log in.": raise Exception("Error logging in.")

        regex =  re.compile("reading work blurb group *") #otherwise get the information about all the fics on the page
        work_list = soup.find("ol", {"class": "reading work index group"}).find_all("li", {"class": regex}) #and put it in a list

        if page == 1: print(f'Starting to pull your data. Currently working on page: 1 out of {pages}') #output that start is successful
        if page % 10 == 0: print(f'Currently working on page: {page} out of {pages}') #output progress every 10 pages read

        for w in work_list: #cycle through each of the works on the page
            if 'Deleted work' not in w.find("h4", {"class": "viewed heading"}).text and 'Mystery' not in w.find("h4", {"class": "heading"}).text: #if the fic isn't removed
                data = getWorkStats(w) #get the information about the work
                ao3History.loc[len(ao3History)] = data #save it
            else:
                if 'Deleted work' in w.find("h4", {"class": "viewed heading"}).text: date = w.find("h4", {"class": "viewed heading"}).text.replace('\n(Deleted work, last visited ','').replace(')\n',''); visitations="times viewed"
                else:
                    date,visitations = w.find("div", {"class": "user module group"}).find("h4").text.split('\nVisited ') #determine when and how often the fic was viewed
                    date = date.replace('\nLast visited: ', '').replace('\n(Latest version.)', '').replace('\n(Minor edits made since then.)','').replace('\n(Update available.)','') #clean up the visitation information (remove other info to just leave the date)
                    visitations = visitations.replace(' times\n', '') #get just the visitation information
                    if(visitations == "once\n"): visitations = 1 #and turn to a number instead of a string
                    else: visitations = int(visitations)
                ao3History.loc[len(ao3History)] = ["title", "url", False, True, "warning", "rating", "ship type", date, date, visitations, "wordcount", "completion status", "authors", "fandoms", "relationships", "characters", "tags", "summary"]

        time.sleep(5) #wait to avoid the timeout message

    print(f'Finished reading {pages} out of {pages}') #alert when finished

In [ ]:
if nativeScrape == 'y' or nativeScrape == 'yes': # if you would like to scrape via the code in this notebook, clean the data you just extracted
    ao3History['updated'] = pd.to_datetime(ao3History['updated'], dayfirst=True, format='mixed')
    ao3History['viewed'] = pd.to_datetime(ao3History['viewed'], dayfirst=True, format='mixed')
    ao3History = ao3History.sort_values('viewed') # sort by date

    saveData = input('Do you want to save your data to a .csv file? (y/n)') #save cleaned data to play with in Excel if that is the user preference
    if saveData == 'y' or saveData == 'yes':
        ao3History.to_csv('ao3CleanHistory.csv')

### (b) Scraping your with another application

If you do not want to enter your password into this code, you can also use a webscraper extension in your browser - I use [Webscraper.io](https://webscraper.io/). It uses the same principles of the webscraping in this code, but is done natively in your browser, so no password entry is necessary! As long as you are already logged into Ao3 in your browser (which you can do before you add the webscraper extension), it should work fine!

To get your data:
1. Install the webscraper extension (note: I am unsure how other webscrapers format their data so you may need to play with certain things if you use another webscraper)
2. Press `Ctrl+Shift+I` on your keyboard to open developer tools and click on the tab `Web Scraper`
3. Click `Create New Sitemap` and select `Import Sitemap` from the dropdown
4. In the box that says `Sitemap JSON` paste the following text (change `USER` and `#PAGES` to your correct info) and import sitemap:

*Code adapted from: __[here](http://www.reddit.com/r/AO3/comments/zu9f5a/ao3_wrapped_history_spreadsheet_how_to_rip_your/)__ by __[u/mrcprlvnn](https://www.reddit.com/user/mrcprlvnn/)__ and __[u/nevereverevee](https://www.reddit.com/user/nevereverevee/):__*
```{"_id":"ao3History","startUrl":["https://archiveofourown.org/users/USER/readings?page=[1-#PAGES]"],"selectors":[{"id":"fic element","multiple":true,"parentSelectors":["_root"],"selector":"li.reading","type":"SelectorElement"},{"id":"url","linkType":"linkFromHref","multiple":false,"parentSelectors":["fic element"],"selector":"h4 a:nth-of-type(1)","type":"SelectorLink"},{"id":"warning","multiple":false,"parentSelectors":["fic element"],"regex":"","selector":".warnings span","type":"SelectorText"},{"id":"rating","multiple":false,"parentSelectors":["fic element"],"regex":"","selector":".rating span","type":"SelectorText"},{"id":"category","multiple":false,"parentSelectors":["fic element"],"regex":"","selector":".category span","type":"SelectorText"},{"id":"posted","multiple":false,"parentSelectors":["fic element"],"regex":"","selector":"p.datetime","type":"SelectorText"},{"id":"viewed","multiple":false,"parentSelectors":["fic element"],"regex":"","selector":"h4.viewed","type":"SelectorText"},{"id":"wordcount","multiple":false,"parentSelectors":["fic element"],"regex":"","selector":"dd.words","type":"SelectorText"},{"id":"completion status","multiple":false,"parentSelectors":["fic element"],"regex":"","selector":".iswip span","type":"SelectorText"},{"id":"author","multiple":true,"parentSelectors":["fic element"],"regex":"","selector":"a:nth-of-type(2)","type":"SelectorText"},{"id":"fandom","multiple":true,"parentSelectors":["fic element"],"regex":"","selector":".fandoms a","type":"SelectorText"},{"id":"relationship","multiple":true,"parentSelectors":["fic element"],"regex":"","selector":".relationships a","type":"SelectorText"},{"id":"character","multiple":true,"parentSelectors":["fic element"],"regex":"","selector":".characters","type":"SelectorText"},{"id":"tags","multiple":true,"parentSelectors":["fic element"],"regex":"","selector":"li:nth-of-type(n+6)","type":"SelectorText"},{"id":"summary","multiple":false,"parentSelectors":["fic element"],"regex":"","selector":".userstuff p","type":"SelectorText"}]}```

5. Go to the `Sitemap ao3History` dropdown and select `Scrape` to start scraping (this can run in the background and you will be notified when it is finished)
6. Go to `Sitemap ao3History` and select `Export Data`, to download the results as a CSV
7. Save the CSV in the same folder as this programme
8. You are now ready to run this part of the code!

In [ ]:
def connect(series):
    """Custom function used with df.groupby() when grouping by title, if each title has multiple different entries.
       Turns the individual entries for each tag/author etc in the dataframe from the same work into one array so each work has a single entry."""
    data = series.astype(str).to_numpy()
    data = [x for x in data if str(x) != 'nan']
    return data

def cleanData(history):
    """A function that will clean the data taken from the online webscraper."""
    history.rename(columns = {'url':'title', 'url-href':'url','category':'ship type', 'posted':'updated', 'author':'authors', 'fandom':'fandoms', 'relationship':'relationships', 'character':'characters'}, inplace = True) #rename column for easier referencing
    history = history.reindex(['title','url','tbr','deleted','warning','rating','ship type','updated','viewed','times viewed','wordcount','completion status','authors','fandoms','relationships','characters','tags','summary'], axis=1) #reorder and add columns to match python webscraping
    history['tbr'] = history['tbr'].astype('boolean'); history['deleted'] = history['deleted'].astype('boolean') #change datatype of new columns
    # get info about deleted works
    history['deleted'].iloc[np.where(history['title'].isnull())] = True
    history['tbr'].iloc[np.where(history['title'].isnull())] = False
    history['deleted'].iloc[np.where(history['deleted'].isna())] = False
    for i in range(len(history)):
        if history['deleted'].iat[i] == True: history['title'].iat[i]=f'deleted work {i}'
    # clean tags info
    history['authors'] = history['authors'][~history.authors.isin(history.fandoms)] #scraping falsly attributed some fandoms as the author and some characters as the normal tags, so clean that up
    history['tags'] = history['tags'][~history.tags.isin(history.characters)]
    history = history.groupby('title').agg({'url':'first', #aggregating data to clean up (as each new tag etc given a new row in the table and each row should be a new work)
                                            'tbr':'first',
                                            'deleted':'first',
                                            'warning':'first',
                                            'rating':'first',
                                            'ship type':'first',
                                            'updated':'first',
                                            'viewed':'first',
                                            'times viewed':'first',
                                            'wordcount':'first',
                                            'completion status':'first',
                                            'authors':connect,
                                            'fandoms':connect,
                                            'relationships':connect,
                                            'characters':connect,
                                            'tags':connect,
                                            'summary':'first'})
    history=history.reset_index() #make it so title is no longer the index
    # clean category information
    history['warning'] = history["warning"].str.split(', ').to_numpy() # change the string for warning and category into an array of values
    history['ship type'] = history["ship type"].str.split(', ').to_numpy()
    history['wordcount'] = history['wordcount'].str.replace(',', '', regex=False) # change wordcount format to integer
    history['wordcount'] = history['wordcount'].astype('Int64')
    # clean the way the date-time data is stored
    for i in range(len(history)):
        viewed = history['viewed'].iloc[i].split('\nVisited ') #determine when and how often the fic was viewed
        history['viewed'].iat[i] = viewed[0].replace('(Deleted work, last visited ','').replace('Last visited: ', '').replace('\n(Latest version.)', '').replace('\n(Minor edits made since then.)','').replace('\n(Update available.)','').replace(')','') #clean up the visitation information (remove other info to just leave the date)
        if type(history['summary'].iat[i]) == str:
            visitations = viewed[1].replace(' times', '') #get just the visitation information
            if 'Marked for Later' in visitations: #note if a work is in your "Marked for Later" or not
                history['tbr'].iat[i] = True
                visitations = visitations.replace('\n(Marked for Later.)', '')
            else: history['tbr'].iat[i] = False
            if(visitations == "once"): history['times viewed'].iat[i] = 1 #and turn to a number instead of a string
            else: history['times viewed'].iat[i] = int(visitations)
    # change format to datetime
    history['updated'] = pd.to_datetime(history['updated'], dayfirst=True, format='mixed')
    history['viewed'] = pd.to_datetime(history['viewed'], dayfirst=True, format='mixed')
    history = history.sort_values('viewed') # sort by date
    return history

In [ ]:
webScrape = input('Have you retrieved your data using a webscraper in a browser extension? (y/n): ') #if someone has hit 'run all' checks you want to perform each part of the code
if webScrape == 'y' or webScrape == 'yes':
    file = input('Please enter the file name (including the .csv extension: ')
    ao3History = pd.read_csv(file).drop(['web-scraper-start-url','web-scraper-order'], axis=1) # import a03 data
    ao3History = cleanData(ao3History)

    saveData = input('Do you want to save your data to a .csv file? (y/n)') #save cleaned data to play with in Excel if that is the user preference
    if saveData == 'y' or saveData == 'yes':
        ao3History.to_csv('ao3CleanHistory.csv')

## Step 2: Creating Functions to Analyse the Data

These are all the functions used to analyse your data. You will need to run these once so that the data can be analysed.

In [ ]:
def numfics(history, date):
    """Gives a general overview of the fics you have read in a given time period."""
    works = len(history[ (history['tbr'] == False) & (history['deleted'] == False) ])
    deleted = len(history[ (history['tbr'] == False) & (history['deleted'] == True) ])
    totalWorks = works + deleted
    print(f'You have read {totalWorks} fics. {deleted/totalWorks*100:.2f}% ({deleted}) of these have been deleted or hidden.')
    print(f"You have read {len(history[history['tbr'] == False])} fics.")
    return history[history['deleted'] == False]

In [ ]:
def tbr(history):
    """Gives a general overview of the fics you have Marked for Later in a given time period."""
    tbr = history[history['tbr'] == True] #pulls the fics Marked for Later

    fandoms = df(columns=['Category', 'Number']) #see top fandoms in tbr
    data=[]
    for i in range(len(tbr['fandoms'])): data.extend(tbr['fandoms'].iloc[i])
    data = np.array(data)
    fandoms['Category'], fandoms['Number'] = np.unique(data, return_counts=True)
    fandoms = fandoms.sort_values(by=['Number'], ascending=False)
    fandoms['Percentage'] = fandoms['Number'].divide(fandoms['Number'].sum(), axis=0)*100

    characters = df(columns=['Category', 'Number']) #see top characters in tbr
    data=[]
    for i in range(len(tbr['characters'])): data.extend(tbr['characters'].iloc[i])
    data = np.array(data)
    characters['Category'], characters['Number'] = np.unique(data, return_counts=True)
    characters = characters.sort_values(by=['Number'], ascending=False)
    characters['Percentage'] = characters['Number'].divide(characters['Number'].sum(), axis=0)*100

    if len(tbr): #if these exist
        sample = tbr.sample().iloc[0]
        print(f'You added {len(tbr)} fics to your Marked for Later list. This is {len(tbr)/len(history)*100:.2f}% of your total works in your history.\n')
        print(f"The top fandom in your Marked for Later is: {fandoms['Category'].iloc[0]} at {fandoms['Percentage'].iloc[0]:.2f}%, followed by {fandoms['Category'].iloc[1]} at {fandoms['Percentage'].iloc[1]:.2f}%.")
        print(f"The top character in your Marked for Later is: {characters['Category'].iloc[0]} at {characters['Percentage'].iloc[0]:.2f}%, followed by {characters['Category'].iloc[1]} at {characters['Percentage'].iloc[1]:.2f}%.")
        print(f"We can see if this is different from your top fandoms and characters you've read this year! \n")
        print(f"A random fic you have in your Marked for Later list is '{sample['title']}' by {sample['authors']} in the {sample['fandoms']} fandom(s). Why not read this fic next?")
        print(f"A summary of this fic is:\n {sample['summary']}")

    else: print(f'You do not have anything in your Marked for Later list. Either you have caught up on everything or you have infinte tabs!')

    tbr = history[history['tbr'] == False]
    tbr['times viewed'] = pd.to_numeric(tbr['times viewed'])
    tbr['wordcount'] = pd.to_numeric(tbr['wordcount'])
    return tbr

In [ ]:
def categoriesOneOnly(history, info):
    """A function that determines the percentage of each rating category in a given time period - for those that only let you choose one."""
    if info == 'rating': colour = 'purple'
    else: colour = 'green'
    categories = history[info].value_counts(ascending=True).reset_index()
    categories.rename(columns = {info:'Category', 'count':'Number'}, inplace = True)
    categories['Percentage'] = categories['Number'].divide(categories['Number'].sum(), axis=0)*100

    #plot the graph
    fig, ax = plt.subplots(figsize=(10,4))
    ax.set_title(info, fontsize=20, fontweight='bold')
    label=categories['Category']
    yticks=np.arange(len(label))
    ax.stem(yticks,categories['Percentage'], linefmt=colour, markerfmt='x', basefmt=colour, orientation='horizontal')
    ax.set_xlabel('Percentage',fontweight='bold')
    ax.set_yticks(yticks, label,fontsize=12, wrap=True)
    ax.set_xlim(0,max(categories['Percentage'])+max(categories['Percentage'])%5+5)
    ax.grid()
    plt.tight_layout()
    plt.show()

    return categories

In [ ]:
def categoriesMany(history, info):
    """A function that determines the percentage of each category in a given time period - for those that let you use multiple (ie. warnings, ship types)."""
    if info == 'warning': colour = 'maroon'
    else: colour = 'deeppink'
    categories = df(columns=['Category','Number']) #initialise the number of each
    data=[] # initialise the data

    for i in range(len(history[info])): data.extend(history[info].iloc[i]) #making the data 1D to better index
    data = np.array(data) #turn to np array
    categories['Category'], categories['Number'] = np.unique(data, return_counts=True) #determine the number
    categories['Percentage'] = categories['Number'].divide(categories['Number'].sum(), axis=0)*100

    #plot the graph
    fig, ax = plt.subplots(figsize=(10,4))
    ax.set_title(info, fontsize=20, fontweight='bold')
    label=categories['Category']
    yticks=np.arange(len(label))
    ax.stem(yticks,categories['Percentage'], linefmt=colour, markerfmt='x', basefmt=colour, orientation='horizontal')
    ax.set_xlabel('Percentage',fontweight='bold')
    ax.set_yticks(yticks, label,fontsize=12, wrap=True)
    ax.set_xlim(0,70)
    ax.grid()
    plt.tight_layout()
    plt.show()

    return categories

In [ ]:
def dates(history, date):
    """A function that plots information about the dates fics were last updated - comparing the dates they were last read and the number read over time."""
    # date read vs date updated
    fig, ax = plt.subplots(figsize=(8,6))
    ax.set_title('Date Read vs Date Updated', fontsize=20, fontweight='bold')
    ax.set_xlabel('Last Read',fontweight='bold')
    ax.set_ylabel('Last Updated',fontweight='bold')
    plt.xticks(rotation=45)
    plt.scatter(history['viewed'], history['updated'], color='teal', s = 0.8)
    ax.grid()
    plt.tight_layout()
    plt.show()

    # fics read over time
    fig, ax = plt.subplots(figsize=(8,6))
    ax.set_title('Number of Fics Read Over Time', fontsize=20, fontweight='bold')
    ax.set_ylabel('Number of Fics Read',fontweight='bold')
    if date == 'all time': ax.set_xlabel('Years',fontweight='bold'); times = np.unique(history['viewed'].dt.year, return_counts=True) #sort by years if looking at all time
    elif len(date.split('-')) == 2: ax.set_xlabel('Days',fontweight='bold'); times = np.unique(history['viewed'].dt.day, return_counts=True) #sort by days if looking at months
    else: ax.set_xlabel('Months',fontweight='bold'); times = np.unique(history['viewed'].dt.month, return_counts=True) #sort by months if looking at a specifc year
    plt.plot(times[0], times[1], color='teal')
    plt.xlim([np.min(times[0]),np.max(times[0])])
    ax.xaxis.set_major_locator(MultipleLocator(1))
    plt.ylim([np.min(times[1])-(np.min(times[1])%5-10),np.max(times[1])+(np.max(times[1])%5+10)])
    ax.grid()
    plt.tight_layout()
    plt.show()

In [ ]:
def visited(history):
    """A function that analyses the number of times you have viewed a fic in a given time period."""
    aveView = history['times viewed'].mean()
    print(f"The fic you viewed the most was {history.sort_values(['times viewed'], ascending=[False])['title'].iloc[0]} by {history.sort_values(['times viewed'], ascending=[False])['authors'].iloc[0]} ({history.sort_values(['times viewed'], ascending=[False])['fandoms'].iloc[0]}) with {history.sort_values(['times viewed'], ascending=[False])['times viewed'].iloc[0]} views.")
    print(f"On average, you view each fic {aveView:.2f} times.")
    return aveView

In [ ]:
def words(history, aveView):
    """A function that analyses the number of words you have read in given time period,
    gives information about the longest, shortest, and average length fics you read,
    and graphs the number of fics read in ranges of word counts as well as words read over time."""
    #general word count analysis
    words = history['wordcount'].sum()
    pages = words/300
    novels =  words/90000
    lotr = words/(187790+ 156198 + 137115)
    print(f'Assuming you have read each fic only once, you have read {words} words.')
    print(f'That is about equal to {pages:.2f} pages, assuming one there are 300 words per page.')
    print(f'Or around {novels:.2f} novel length books (90,000 words per novel). That means you could have read the Lord of the Rings trilogy {lotr:.2f} times.\n')
    print(f'Taking into account that you viewed each fic on average {aveView:.2f} times, and assuming you fully reread the fic each time you viewed it:')
    print(f'you read {int(np.round(words*aveView))} words, for {pages*aveView:.2f} pages or {novels*aveView:.2f} novels (thats LotR {lotr*aveView:.2f} times).\n')
    print(f"The longest fic you have read was {history.sort_values(['wordcount'], ascending=[False])['title'].iloc[0]} by {history.sort_values(['wordcount'], ascending=[False])['authors'].iloc[0]} ({history.sort_values(['wordcount'], ascending=[False])['fandoms'].iloc[0]}), at {history.sort_values(['wordcount'], ascending=[False])['wordcount'].iloc[0]} words.")
    print(f"The shortest fic you have read was {history.sort_values(['wordcount'], ascending=[True])['title'].iloc[0]} by {history.sort_values(['wordcount'], ascending=[True])['authors'].iloc[0]} ({history.sort_values(['wordcount'], ascending=[True])['fandoms'].iloc[0]}), at {history.sort_values(['wordcount'], ascending=[True])['wordcount'].iloc[0]} words.")
    print(f"The average length fanfic you read is {history['wordcount'].mean():.2f} words.")

    #grouping the word counts read into discrete categories to determine the number of fics read in each category (categories not evenly defined)
    startbins = [0,100,500,1000,2000,5000,10000,20000,50000,100000,500000,1000000]
    words = history.groupby(pd.cut(history['wordcount'], startbins), observed=False).agg({'wordcount':['mean', 'count']})
    words.columns = words.columns.droplevel(0)
    words = words.reset_index()
    words['Percentage'] = words['count'].divide(words['count'].sum(), axis=0)*100

    #plotting the number of fics read in eah word count category
    fig, ax = plt.subplots(figsize=(10,5))
    ax.set_title('Percentage of Fics Read vs Wordcount', fontsize=20, fontweight='bold')
    label=words['wordcount']
    yticks=np.arange(len(label))
    ax.stem(yticks,words['Percentage'], linefmt='springgreen', markerfmt='x', basefmt='springgreen', orientation='horizontal')
    ax.set_xlabel('Percentage of Fics Read',fontweight='bold')
    ax.set_ylabel('Range of Words')
    ax.set_yticks(yticks, label,fontsize=12, wrap=True)
    ax.set_xlim(0,30)
    ax.grid()
    plt.tight_layout()
    plt.show()

    #plotting the words read over time (similar to fics read over time)
    wordsInTime = []
    fig, ax = plt.subplots(figsize=(8,6))
    ax.set_title('Number of Words Read Over Time', fontsize=20, fontweight='bold')
    ax.set_ylabel('Number of Words Read',fontweight='bold')
    if date == 'all time': ax.set_xlabel('Years',fontweight='bold'); dates, indices = np.unique(history['viewed'].dt.year, return_index=True)
    elif len(date.split('-')) == 2: ax.set_xlabel('Days',fontweight='bold'); dates, indices = np.unique(history['viewed'].dt.day, return_index=True)
    else: ax.set_xlabel('Months',fontweight='bold'); dates, indices = np.unique(history['viewed'].dt.month, return_index=True)
    for i in range(len(indices)-1): wordsInTime.append( history['wordcount'].iloc[indices[i]:indices[i+1]].sum() )
    wordsInTime.append( history['wordcount'].iloc[indices[-1]:-1].sum() )
    plt.plot(dates, wordsInTime, color='springgreen')
    plt.xlim([np.min(dates),np.max(dates)])
    ax.xaxis.set_major_locator(MultipleLocator(1))
    plt.ylim([np.min(wordsInTime)-(np.min(wordsInTime)%5)-10,np.max(wordsInTime)+(np.max(wordsInTime)%5)+10])
    ax.grid()
    plt.tight_layout()
    plt.show()

    return words

In [ ]:
def topInfo(history, information):
    """A function that determines the number of times you have read works from each data type in your history in a certain time period.
    Plots the percentage of fics you have read that make up your top 20 info."""
    #sort and group data according to info instead of works
    info = df(columns=['Category', 'Number'])
    data=[]
    for i in range(len(history[information])): data.extend(history[information].iloc[i])
    data = np.array(data)
    info['Category'], info['Number'] = np.unique(data, return_counts=True)
    info = info.sort_values(by=['Number'], ascending=False)
    info['Percentage'] = info['Number'].divide(info['Number'].sum(), axis=0)*100

    #output information about your top information for the category
    print(f'You have read fics with {len(info)} different {information}. \n')
    n = 20
    percentage = info['Percentage'].head(n).sum()
    print(f'Your top {n} {information} were in {percentage:.2f}% of your read fics.')
    print(f'Of your top {n} {information}, the percentage of fics you read with them are:')

    #plot your top information for the category
    fig, ax = plt.subplots(figsize=(11,10))
    ax.set_title(information, fontsize=20, fontweight='bold')
    label=info['Category'].head(n)
    yticks=np.flip(np.arange(len(label)))
    ax.stem(yticks,info['Percentage'].head(n), linefmt='dodgerblue', markerfmt='b x', basefmt='dodgerblue', orientation='horizontal')
    ax.set_xlabel('Percentage',fontweight='bold')
    ax.set_xlim(0,np.max(info['Percentage'].head(n))+np.max(info['Percentage'].head(n))%2)
    ax.xaxis.set_major_locator(MultipleLocator(2))
    ax.xaxis.set_minor_locator(MultipleLocator(.5))
    ax.grid(which='both', axis='both')
    ax.set_yticks(yticks, label,fontsize=10, wrap=True)
    plt.tight_layout()
    plt.show()
    return info

In [ ]:
### INTERACTIVE PLOTS (SEE CHANGING INFO OVER TIME)

def getInfo(date, history, info):
    """A function that pulls information divided into years, months or days depending on the selected time period, and bins it depending on the number of unique values."""
    infoInTime = []
    #get information based on time period
    if date == 'all time': times, indices = np.unique(history['viewed'].dt.year, return_index=True)
    elif len(date.split('-')) == 2: times, indices = np.unique(history['viewed'].dt.day, return_index=True)
    else: times, indices = np.unique(history['viewed'].dt.month, return_index=True)
    for i in range(len(indices)-1): infoInTime.append( history[info].iloc[indices[i]:indices[i+1]].sum() )
    infoInTime.append( history[info].iloc[indices[-1]:-1].sum() )
    #bin the data to get the counts of each unique information
    infoBinned = df(columns=['Date', 'Category', 'Number'])
    for i in range(len(times)):
        cat, num = np.unique(infoInTime[i], return_counts=True)
        infoBinned.loc[len(infoBinned)] = [times[i], cat, num]
    return infoBinned, times

def getInfoAtDate(chosenDate, infoBinned):
    """A function that dynamically picks the current chosen date to pull the information from just that date."""
    global times
    readToday = True
    infoDated = df(columns=['Category', 'Number'])
    if True in (times == chosenDate):
        infoDated['Category'] = infoBinned['Category'].loc[np.where(times == chosenDate)[0][0]]
        infoDated['Number'] = infoBinned['Number'].loc[np.where(times == chosenDate)[0][0]]
        infoDated = infoDated.sort_values(by=['Number'], ascending=False)
        infoDated['Percentage'] = infoDated['Number'].divide(infoDated['Number'].sum(), axis=0)*100
    else: readToday = False
    return readToday, infoDated

def compareTopChanges(chosenDate, times, infoBinned, info, infoToday, rank):
    """A function that looks at the top information and how it has changed"""
    readYesterday, infoYesterday = getInfoAtDate(times[np.where(times == chosenDate)[0][0]-1], infoBinned)
    infoToday = infoToday.reset_index(drop=True)
    infoYesterday = infoYesterday.reset_index(drop=True)
    topChange = infoYesterday[infoYesterday['Category'] == infoToday['Category'].iloc[rank]]
    if topChange.empty == True: print(f"- {infoToday['Category'].iloc[rank]} is a new to your read {info} and you haven't read a fic for this in the previous time period.")
    else:
        index = topChange.index[0]
        print(f"- {infoToday['Category'].iloc[rank]} went up {index-rank} places from the previous time period, from {topChange['Number'].iloc[0]} fic(s) ({topChange['Percentage'].iloc[0]:.2f}%) to {infoToday['Number'].iloc[rank]} fic(s) ({infoToday['Percentage'].iloc[rank]:.2f}%).")

def getRanks(chosenDate, times, infoBinned, info, infoToday, data):
    """A function that allows the user to find the rank of a specific searched term and how it has changed."""
    readYesterday, infoYesterday = getInfoAtDate(times[np.where(times == chosenDate)[0][0]-1], infoBinned)
    infoToday = infoToday.reset_index(drop=True)
    infoYesterday = infoYesterday.reset_index(drop=True)
    indices = np.array(np.where(infoToday['Category'].str.contains(data, case=False))).flatten()
    if len(indices) == 0 : print(f"I couldn't find a {info} that contains this during this date!")
    else:
        for i in range(len(indices)):
            topChange = infoYesterday[infoYesterday['Category'] == infoToday['Category'].iloc[indices[i]]]
            if topChange.empty == True: print(f"- {infoToday['Category'].iloc[indices[i]]} is a new to your read {info} and you haven't read a fic for this in the previous time period.")
            else:
                index = topChange.index[0]
                print(f"- {infoToday['Category'].iloc[indices[i]]} is at rank {indices[i]} and went up {index-indices[i]} places from the previous time period, from {topChange['Number'].iloc[0]} fic(s) ({topChange['Percentage'].iloc[0]:.2f}%) to {infoToday['Number'].iloc[indices[i]]} fic(s) ({infoToday['Percentage'].iloc[indices[i]]:.2f}%).")

def plot_func(chosenDate, info, infoBinned):
    """A function that calls 'getInfoAtDate' to dynamically get information from that date and then plots the top values of it."""
    readToday,infoDated = getInfoAtDate(chosenDate, infoBinned)
    if readToday == False: print('You did not read anything during this time period! Select a different date!')
    else:
        if chosenDate != times[0]:
            print(f"For your top {info}:")
            compareTopChanges(chosenDate, times, infoBinned, info, infoDated, 0)
            print(f"For your 2nd most read {info}")
            compareTopChanges(chosenDate, times, infoBinned, info, infoDated, 1)
            print(f"For your 3rd {info}")
            compareTopChanges(chosenDate, times, infoBinned, info, infoDated, 2)
            print(f"For your 4th {info}")
            compareTopChanges(chosenDate, times, infoBinned, info, infoDated, 3)
            print(f"For your 5th {info}")
            compareTopChanges(chosenDate, times, infoBinned, info, infoDated, 4)

        print(f'\n Choose an {info} you would like to find a rank for: ')
        interact(getRanks,
                 data=widgets.Text(value=f'{info}',placeholder=f'{info}',description=f'{info} name: ', disabled=False),
                 chosenDate=fixed(chosenDate),times=fixed(times), infoBinned=fixed(infoBinned), info=fixed(info), infoToday=fixed(infoDated))

        fig, ax = plt.subplots(figsize=(11,4.5))
        ax.set_title(f'{info} At Date: {chosenDate}', fontsize=20, fontweight='bold')
        label=infoDated['Category'].head(10)
        yticks=np.flip(np.arange(len(label)))
        xvalues = infoDated['Percentage'].head(10)
        ax.stem(yticks, xvalues, linefmt='peru', markerfmt='y x', basefmt='peru', orientation='horizontal')
        ax.set_xlabel('Percentage',fontweight='bold')
        ax.set_xlim(0,np.max(infoDated['Percentage'].head(10))+np.max(infoDated['Percentage'].head(10))%2)
        ax.grid(which='both', axis='both')
        ax.set_yticks(yticks, label,fontsize=10, wrap=True)
        plt.tight_layout()
        plt.show()

## Step 3: Run the Analysis to Look At Your Data!

First you need to get your dataframe - if this is the first time you have run this notebook, you have pulled the dataframe using the previous code! However, if you have run this code before, and would like to just look at your stats again without scraping your most recent data (especially as that process can take a while depending on your history!) then you need to run the first cell.

Then the following cells can be run multiple times, to look at many different time periods. You can look at your stats for all time, by year, or by month in a specific year.
- you can run all of the following cells to start with if you have not run all (you will definitely need to run cell 3 once)
- if you want to check another time period, you must rerun **cell 4** (to change the time period), **cell 5** (to remove the deleted works from your analysis), and **cell 6** (to remove the works on your Marked for Later list from the analysis) in order - **otherwise, your stats will not update**
- any of the other cells you can run as many times as you want in any order you would like to look at the stats of interest to you!

Please note - due to the limitations of how Ao3 records your history (ie. each fic is only output once in your history at the last time you read a fic), the further back in history you go, the more inaccurate it may be, as fics you have read in previous years will now only show as being read in the latest year you read them. This may especially affect any analysis you do of previous years. Also note that your history page only shows when a fic was last updated, not when it was first posted (unlike the fic page itself) so that is the date that is used.

Once you have picked a time period, you can:
- sample a random fic you read from that time period
- look at the total number of fics you read, and what percentage of those have been deleted
- look at how many works are on your 'Marked for Later' list and see information (top fandom and top characters) about them
- show you graphs of percentage of fics you read with each type of warning, rating, and ship type, as well as your ratio of completed works read
- show you the last time you viewed a fic compared with the last time it was updated
- the number of fics read over time (sorted by years, months or days depending on whether you are looking at all time, a certain year, or certain month respectively)
- analyse the number of words you read and words read over time (similar to fics read over time)
- identify your top 20 authors, fandoms, relationships, characters and tags
- dynamically choose and plot the changes to your top 10 authors, fandoms, relationships, characters and tags over the time period (by years, months or days depending on the inital overview, similar to fics read over time), see how the rankings of your top 5 have changed from the previous time period, and check the ranking of a specified author/fandom/ship/character/tag

In [ ]:
# if you are retrieving pre-saved data run this cell!
savedData = input('Have you run the previous cells? (If yes, you can use the data you just scraped, otherwise, you need to load a pre-saved dataframe). (y/n): ')
if savedData == 'y' or savedData == 'yes': pass
else:
    file = input('Please enter the file name (including the .csv extension: ')
    ao3History = pd.read_csv(file, index_col='Unnamed: 0')
    for i in range(len(ao3History)):
        if '[' in ao3History['warning'].iat[i]: ao3History['warning'].iat[i] = literal_eval(ao3History['warning'].iat[i])
        if '[' in ao3History['ship type'].iat[i]: ao3History['ship type'].iat[i] = literal_eval(ao3History['ship type'].iat[i])
        if '[' in ao3History['authors'].iat[i]: ao3History['authors'].iat[i] = literal_eval(ao3History['authors'].iat[i])
        if '[' in ao3History['fandoms'].iat[i]: ao3History['fandoms'].iat[i] = literal_eval(ao3History['fandoms'].iat[i])
        if '[' in ao3History['relationships'].iat[i]: ao3History['relationships'].iat[i] = literal_eval(ao3History['relationships'].iat[i])
        if '[' in ao3History['characters'].iat[i]: ao3History['characters'].iat[i] = literal_eval(ao3History['characters'].iat[i])
        if '[' in ao3History['tags'].iat[i]: ao3History['tags'].iat[i] = literal_eval(ao3History['tags'].iat[i])
    ao3History['updated'] = pd.to_datetime(ao3History['updated'], dayfirst=True, format='mixed')
    ao3History['viewed'] = pd.to_datetime(ao3History['viewed'], dayfirst=True, format='mixed')

In [ ]:
ao3History.sort_values('viewed') #show table

In [ ]:
def filterdb(history):
    global date
    date = input("What time period would you like to filter for? \n You may choose: 'all time', type a specific year (format yyyy), or select a month (format mm-yyyy): ")
    if date == 'all time': return history
    elif len(date.split('-')) == 2: return history[( history['viewed'].dt.month == int(date.split('-')[0]) ) & ( history['viewed'].dt.year == int(date.split('-')[1]) )]
    else: return history[history['viewed'].dt.year == int(date)]

In [ ]:
filtered = filterdb(ao3History)

In [ ]:
print(f'Your stats for {date} are: \n')
notDeleted = numfics(filtered, date)

In [ ]:
def getTags(history, info, text):
    containsTrueFalse = pd.Series(index=history.index, name=info, dtype='bool')
    for index,row in history.iterrows():
        for word in row[info]:
            if text.lower() in word.lower():
                containsTrueFalse.iloc[index]=True
            else: containsTrueFalse.iloc[index]=False
    return containsTrueFalse

def filterDB(history, haveTit, haveTbr, haveWarn, haveRate, haveCat, haveWord, haveComplete, haveAuthor, haveShips, haveFandoms, haveCharac, haveTags, haveSumm):
    titletext = history['title'].str.contains(haveTit, case=False)
    if haveTbr==True: tbrtext = history['tbr'] == True
    else: tbrtext = history['tbr'] == False
    wordtext = (history['wordcount'].astype('int') > haveWord[0]) & (history['wordcount'].astype('int') < haveWord[1])
    warntext = getTags(history, 'warning', haveWarn)
    ratetext = history['rating'] == haveRate
    cattext = getTags(history, 'ship type', haveCat)
    completetext = history['completion status'].str.contains(haveComplete, case=False)
    authtext = getTags(history, 'authors', haveAuthor)
    shiptext = getTags(history, 'relationships', haveShips)
    fandomtext = getTags(history, 'fandoms', haveFandoms)
    charactext = getTags(history, 'characters', haveCharac)
    tagtext = getTags(history, 'tags', haveTags)
    summtext = history['summary'].str.contains(haveSumm, case=False)
    return history[titletext & tbrtext & warntext & ratetext & cattext & wordtext & completetext & authtext & shiptext & fandomtext & charactext & tagtext & summtext]

wordrange = np.arange(np.min(notDeleted['wordcount'].astype('int')),np.max(notDeleted['wordcount'].astype('int')),1000)

print('Wold you like to filter for a fic by: ')
interact(filterDB, history=fixed(ao3History[ao3History['deleted']==False].reset_index(drop=True)),
                   haveTit=widgets.Text(value='',placeholder='text in title',description='the title containing:',disabled=False),
                   haveTbr=widgets.Checkbox(value=False,description='in your Marked for Later?',disabled=False),
                   haveWarn=widgets.Select(options=['Choose Not To Use Archive Warnings', 'Graphic Depictions Of Violence', 'Major Character Death', 'No Archive Warnings Apply', 'Rape/Non-Con', 'Underage'],description='warning?',disabled=False),
                   haveRate=widgets.Select(options=['Not Rated', 'General Audiences', 'Teen and Up Audiences', 'Mature ', 'Explicit'],description='warning?',disabled=False),
                   haveCat=widgets.Select(options=['F/F', 'F/M', 'Gen', 'M/M ', 'Multi', 'Other'],description='ship type (category)?',disabled=False),
                  haveWord=widgets.SelectionRangeSlider(options=wordrange,index=(0,len(wordrange)-1),description='# Words',disabled=False),
                   haveComplete=widgets.Dropdown(options=['Complete Work', 'Work in Progress'], description='Category:', disabled=False),
                   haveAuthor=widgets.Text(value='',placeholder='text in author',description='the author containing:',disabled=False),
                   haveShips=widgets.Text(value='',placeholder='text in ships',description='the ships containing:',disabled=False),
                   haveFandoms=widgets.Text(value='',placeholder='text in fandoms',description='the fandoms containing:',disabled=False),
                   haveCharac=widgets.Text(value='',placeholder='text in characters',description='the characters containing:',disabled=False),
                   haveTags=widgets.Text(value='',placeholder='text in tags',description='the tags containing:',disabled=False),
                   haveSumm=widgets.Text(value='',placeholder='text in summary',description='the title containing:',disabled=False))

In [ ]:
notTbr = tbr(notDeleted)

In [ ]:
print('A random fic you read was: \n')
filtered.sample()

In [ ]:
warningstats = categoriesMany(notTbr, 'warning')

In [ ]:
ratingstats =  categoriesOneOnly(notTbr, 'rating')

In [ ]:
catstats = categoriesMany(notTbr, 'ship type')

In [ ]:
completedstats = categoriesOneOnly(notTbr, 'completion status')

In [ ]:
dates(notTbr, date)

In [ ]:
aveView = visited(notTbr)

In [ ]:
wordstats = words(notTbr, aveView)

In [ ]:
authstats = topInfo(notTbr, 'authors')
print(authstats.head(n=20))
print(f'\n Change your chosen date so see how your top authors have changed over the selected time period.')

authorsBinned, times = getInfo(date, notTbr, 'authors')
interact(plot_func, chosenDate=widgets.IntSlider(value=1, min=times[0], max=times[-1], step=1), info=fixed('authors'), infoBinned=fixed(authorsBinned));

In [ ]:
fandomstats = topInfo(notTbr, 'fandoms')
print(fandomstats.head(n=20))
print(f'\n Change your chosen date so see how your top fandoms have changed over the selected time period.')

fandomsBinned, times = getInfo(date, notTbr, 'fandoms')
interact(plot_func, chosenDate=widgets.IntSlider(value=1, min=times[0], max=times[-1], step=1), info=fixed('fandoms'), infoBinned=fixed(fandomsBinned));

In [ ]:
shipstats = topInfo(notTbr,'relationships')
print(shipstats.head(n=20))
print(f'\n Change your chosen date so see how your top ships have changed over the selected time period.')

shipsBinned, times = getInfo(date, notTbr, 'relationships')
interact(plot_func, chosenDate=widgets.IntSlider(value=1, min=times[0], max=times[-1], step=1), info=fixed('relationships'), infoBinned=fixed(shipsBinned));

In [ ]:
charstats = topInfo(notTbr, 'characters')
print(charstats.head(n=20))
print(f'\n Change your chosen date so see how your top characters have changed over the selected time period.')

charactersBinned, times = getInfo(date, notTbr, 'characters')
interact(plot_func, chosenDate=widgets.IntSlider(value=1, min=times[0], max=times[-1], step=1), info=fixed('characters'), infoBinned=fixed(charactersBinned));

In [ ]:
tagstats = topInfo(notTbr, 'tags')
print(tagstats.head(n=20))
print(f'\n Change your chosen date so see how your top tags have changed over the selected time period.')

tagsBinned, times = getInfo(date, notTbr, 'tags')
interact(plot_func, chosenDate=widgets.IntSlider(value=1, min=times[0], max=times[-1], step=1), info=fixed('tags'), infoBinned=fixed(tagsBinned));